# CausalProbe Paper Table (Pretty Version)

This notebook builds a cleaner table for the 8 final result files:

- E / H
- vanilla / CoT / RAG / G²

It assumes you will rename the merged files back to the final normal output names, so this version does **not** special-case merged/rerun files.


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
RESULT_DIR_E = PROJECT_ROOT / 'result_logs' / 'E'
RESULT_DIR_H = PROJECT_ROOT / 'result_logs' / 'H'

# Update these only if your final file names differ.
FILES = {
    ('E', 'Vanilla'): RESULT_DIR_E / 'result_deepseek-v3.1_671b-cloud_vanilla_CausalProbe-E_mcqa_causalprobe.json',
    ('E', 'CoT'):     RESULT_DIR_E / 'result_deepseek-v3.1_671b-cloud_vanilla_CausalProbe-E_mcqa_cot_causalprobe.json',
    ('E', 'RAG'):     RESULT_DIR_E / 'result_deepseek-v3.1_671b-cloud_retrieval_CausalProbe-E_mcqa_retrieval_causalprobe.json',
    ('E', 'G²'):      RESULT_DIR_E / 'result_deepseek-v3.1_671b-cloud_retrieval_CausalProbe-E_mcqa_g2reasoner_causalprobe.json',

    ('H', 'Vanilla'): RESULT_DIR_H / 'result_deepseek-v3.1_671b-cloud_vanilla_CausalProbe-H_mcqa_causalprobe.json',
    ('H', 'CoT'):     RESULT_DIR_H / 'result_deepseek-v3.1_671b-cloud_vanilla_CausalProbe-H_mcqa_cot_causalprobe.json',
    ('H', 'RAG'):     RESULT_DIR_H / 'result_deepseek-v3.1_671b-cloud_retrieval_CausalProbe-H_mcqa_retrieval_causalprobe.json',
    ('H', 'G²'):      RESULT_DIR_H / 'result_deepseek-v3.1_671b-cloud_retrieval_CausalProbe-H_mcqa_g2reasoner_causalprobe.json',
}

FILES


In [ ]:
def load_jsonl(path: Path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def summarize_result(path: Path):
    rows = load_jsonl(path)
    total = len(rows)
    correct = sum(r.get('metric_result') is True for r in rows)
    no_choice = sum('No choice id is outputted' in str(r.get('metric_result', '')) for r in rows)
    err_429 = sum('429 Client Error' in str(r.get('output', '')) for r in rows)
    err_500 = sum('500 Server Error' in str(r.get('output', '')) for r in rows)
    valid = sum(r.get('metric_result') is True or r.get('metric_result') is False for r in rows)

    return {
        'total': total,
        'correct': correct,
        'accuracy': correct / total if total else None,
        'valid_answers': valid,
        'valid_accuracy': correct / valid if valid else None,
        'no_choice': no_choice,
        'err_429': err_429,
        'err_500': err_500,
    }


In [ ]:
records = []

for (dataset, method), path in FILES.items():
    if not path.exists():
        records.append({
            'dataset': dataset,
            'method': method,
            'file': str(path),
            'missing': True,
        })
        continue

    s = summarize_result(path)
    records.append({
        'dataset': dataset,
        'method': method,
        'file': str(path),
        'missing': False,
        **s,
    })

detail_df = pd.DataFrame(records)
detail_df


In [ ]:
# Main paper table: accuracy only
paper_table = detail_df[detail_df['missing'] == False].pivot(index='dataset', columns='method', values='accuracy')
paper_table = paper_table.reindex(index=['E', 'H'], columns=['Vanilla', 'CoT', 'RAG', 'G²'])
paper_table = paper_table.applymap(lambda x: round(x, 4) if pd.notna(x) else x)
paper_table


In [ ]:
# Pretty percent display for report drafting
pretty_table = paper_table.copy()
for col in pretty_table.columns:
    pretty_table[col] = pretty_table[col].map(lambda x: f'{x*100:.2f}%' if pd.notna(x) else 'MISSING')
pretty_table


In [ ]:
# Diagnostic table: useful for appendix / sanity checking
diag_cols = ['dataset', 'method', 'total', 'correct', 'accuracy', 'no_choice', 'err_429', 'err_500', 'file']
diagnostic_table = detail_df[[c for c in diag_cols if c in detail_df.columns]].copy()
diagnostic_table = diagnostic_table.sort_values(['dataset', 'method'])
diagnostic_table


In [ ]:
# Save outputs
out_dir = PROJECT_ROOT / 'result_logs'
out_dir.mkdir(exist_ok=True)

paper_csv = out_dir / 'paper_table_accuracy.csv'
pretty_csv = out_dir / 'paper_table_accuracy_pretty.csv'
diagnostic_csv = out_dir / 'paper_table_diagnostic.csv'

paper_table.to_csv(paper_csv)
pretty_table.to_csv(pretty_csv)
diagnostic_table.to_csv(diagnostic_csv, index=False)

print('Saved:', paper_csv)
print('Saved:', pretty_csv)
print('Saved:', diagnostic_csv)
